# Panel de préstamos por actividad económica

Apila los archivos del BCRA "Préstamos por actividad económica" disponibles en `data/raw/bcra_prestamos_actividad/` (33 archivos: act, actgrup, loc para 2015-2025). Las series son trimestrales.

Genera **tres paneles separados** por su estructura distinta:
- `panel_actividad_total.parquet` (de `act{YYYY}.xls`): saldo por actividad económica × moneda × trimestre, total sistema.
- `panel_actividad_grupo.parquet` (de `act{YYYY}grup.xls`): mismo desglose pero por grupo de entidad.
- `panel_actividad_localidad.parquet` (de `loc{YYYY}.xls`): saldo por provincia/departamento × moneda × trimestre.

**Estos paneles preservan el formato wide del BCRA** (las columnas `act00t`, `act01t`, ... son distintas categorías de préstamo definidas por el BCRA, no fechas). El crosswalk de qué representa cada `actXX` se construye en notebook de análisis posterior según necesidad — la documentación oficial está en la hoja `Observaciones` de cada xls.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

ACT_ROOT = RAW / "bcra_prestamos_actividad"
OUT_TOTAL = PANELES / "panel_actividad_total.parquet"
OUT_GRUPO = PANELES / "panel_actividad_grupo.parquet"
OUT_LOC = PANELES / "panel_actividad_localidad.parquet"
OUT_TOTAL.parent.mkdir(parents=True, exist_ok=True)

YEARS = range(2015, 2026)
HEADER_ROW_DATA = 25  # los xls tienen 25 filas de encabezado antes de los datos

## Función de lectura

Los xls del BCRA tienen ~25 filas de cabecera mezclando bilingüe y notas. La fila 25 trae los nombres de columna efectivos (tipo `orden`, `name`, `ciiu`, `actfec`, `actmon`, `act00t`...). Los datos comienzan en la fila 26.

In [ ]:
def leer_actividad(filepath):
    df = pd.read_excel(filepath, sheet_name=0, header=HEADER_ROW_DATA, engine="xlrd")
    df = df.dropna(how="all")
    return df

def apilar_anios(prefix):
    """prefix in ('act', 'actgrup', 'loc')"""
    bloques = []
    for y in YEARS:
        fname_map = {"act": f"act{y}.xls", "actgrup": f"act{y}grup.xls", "loc": f"loc{y}.xls"}
        f = ACT_ROOT / fname_map[prefix]
        if not f.exists():
            continue
        df = leer_actividad(f)
        df["anio_archivo"] = y
        bloques.append(df)
    return pd.concat(bloques, ignore_index=True)

## Apilado

In [ ]:
total = apilar_anios("act")
grupo = apilar_anios("actgrup")
loc   = apilar_anios("loc")
print(f"Total:     {len(total):,} filas, {len(total.columns)} cols")
print(f"Grupo:     {len(grupo):,} filas, {len(grupo.columns)} cols")
print(f"Localidad: {len(loc):,} filas, {len(loc.columns)} cols")

## Persistencia

In [ ]:
total.to_parquet(OUT_TOTAL, index=False)
grupo.to_parquet(OUT_GRUPO, index=False)
loc.to_parquet(OUT_LOC, index=False)
print(f"Escrito: {OUT_TOTAL.relative_to(REPO)}")
print(f"Escrito: {OUT_GRUPO.relative_to(REPO)}")
print(f"Escrito: {OUT_LOC.relative_to(REPO)}")

## Validaciones

In [ ]:
for nombre, df in [("total", total), ("grupo", grupo), ("localidad", loc)]:
    assert len(df) > 0, f"Panel {nombre} vacío"
    assert df["anio_archivo"].nunique() == len(YEARS), f"Faltan años en {nombre}"
print("Validaciones OK")

## Muestra: estructura del panel total

In [ ]:
duckdb.sql(f"select * from '{OUT_TOTAL}' limit 3").df()